In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, LineString

In [3]:

# 1. Configuration
ROOT_DIR = '../02_INTERMEDIATE/'  # Point this to your master folder path
TARGET_FILE_NAME = 'individual_bamboo_params_breast_height_v1.gpkg'
OUTPUT_FILE_PATH = '../02_INTERMEDIATE/merged_drone_bamboo_params_breast_height.gpkg'

In [4]:
def calculate_canopy_diameter(geometry):
    """
    Calculates the maximum diameter (longest distance between any two vertices) 
    of a geometry using its Minimum Bounding Rectangle.
    """
    if geometry is None or geometry.is_empty:
        return 0.0
    
    # Enclose the clump boundary using an oriented/rotated bounding box
    mbr = geometry.minimum_rotated_rectangle
    
    if isinstance(mbr, Polygon):
        coords = list(mbr.exterior.coords)
        
        # Calculate orthogonal side lengths of the minimum bounding box
        side1 = np.sqrt((coords[1][0] - coords[0][0])**2 + (coords[1][1] - coords[0][1])**2)
        side2 = np.sqrt((coords[2][0] - coords[1][0])**2 + (coords[2][1] - coords[1][1])**2)
        
        # The longest possible line segment within the bounding box is its diagonal
        diagonal = np.sqrt(side1**2 + side2**2)
        return float(diagonal)
        
    elif isinstance(mbr, LineString):
        return float(mbr.length)
    else:
        minx, miny, maxx, maxy = geometry.bounds
        return float(np.sqrt((maxx - minx)**2 + (maxy - miny)**2))

In [5]:
# 2. Extract PSP subfolders and file paths
gpkg_files = []
for dirpath, _, filenames in os.walk(ROOT_DIR):
    if TARGET_FILE_NAME in filenames:
        gpkg_files.append(os.path.join(dirpath, TARGET_FILE_NAME))

In [6]:
# 3. Process, append calculations, and archive
gdf_list = []
for file_path in gpkg_files:
    # Extract the name of the folder containing the file as the PSP ID
    psp_id = os.path.basename(os.path.dirname(file_path))
    
    gdf = gpd.read_file(file_path)
    
    # Process calculations 
    gdf['canopy_diameter'] = gdf['geometry'].apply(calculate_canopy_diameter)
    gdf['psp_id'] = psp_id
    
    # Save updates directly into the individual subfolder files
    gdf.to_file(file_path, driver='GPKG')
    
    gdf_list.append(gdf)

In [7]:

# 4. Standardize Coordinate Systems and Merge
if gdf_list:
    target_crs = gdf_list[0].crs
    
    # Reproject layer datasets dynamically if spatial reference systems mismatch across plots
    standardized_gdfs = [g.to_crs(target_crs) if g.crs != target_crs else g for g in gdf_list]
    
    merged_gdf = gpd.GeoDataFrame(pd.concat(standardized_gdfs, ignore_index=True), crs=target_crs)
    merged_gdf.to_file(OUTPUT_FILE_PATH, driver='GPKG')
    print(f"Success! Master compilation file saved to {OUTPUT_FILE_PATH}")
    print(f"Total PSP plots processed: {len(gdf_list)}")
else:
    print(f"No files matching '{TARGET_FILE_NAME}' were found under the specified root directory.")

Success! Master compilation file saved to ../02_INTERMEDIATE/merged_drone_bamboo_params_breast_height.gpkg
Total PSP plots processed: 7
